# SqueezeNet

This notebook explains how to train and evaluate the SqueezeNet model on the FLIM datasets: eggs, larvae and cysts.

SqueezeNet is a family of convolutional neural networks introduced by Iandola et al. (2016). It achieves AlexNet-level accuracy on ImageNet with 50x fewer parameters by using Fire modules, which are composed of a squeeze layer with 1×1 convolutions feeding into an expand layer with a mix of 1×1 and 3×3 convolutions. This design drastically reduces model size, making SqueezeNet well suited for deployment in memory-constrained environments.

Paper: Iandola, F., Han, S., Moskewicz, M., Ashraf, K., Dally, W. & Keutzer, K. (2016). SqueezeNet: AlexNet-level accuracy with 50x fewer parameters and less than 0.5MB model size. https://arxiv.org/abs/1602.07360

PyTorch implementation: this notebook uses the `torchvision.models` implementation named `squeezenet1_0`. The implementation supports loading
ImageNet pretrained weights and adapting the final classification layer for our tasks. See the PyTorch documentation for details: https://docs.pytorch.org/vision/main/models/squeezenet.html

What you can do in this notebook: train from scratch with random initialization or fine-tune from ImageNet pretrained weights. Configure dataset,
training schedule and other hyperparameters in the `CONFIG_*` blocks.

Quick start:
1. Activate the virtual environment: `source venv/bin/activate`
2. Install dependencies: `pip install -r requirements.txt`
3. Run the cells in order. Edit the `CONFIG_*` blocks to set dataset, data fraction, number of epochs and other options.

Notes:
Paths in this notebook are configured relative to the repository root using `NOTEBOOK_DIR`, `BASELINE_DIR` and `PROJECT_ROOT`.
Results, model checkpoints and JSON reports are written to folders such as `squeezenet/squeezenet_scratch/eggs` or `squeezenet/squeezenet_pretrained/eggs` depending on the chosen mode.

In [ ]:
# Imports and notebook setup
import sys
from pathlib import Path
import json

# Core PyTorch stack
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision import transforms
from torchinfo import summary

# Data handling and visualization
import os
import seaborn as sns
import PIL
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import cohen_kappa_score

# Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Repository paths
NOTEBOOK_DIR = Path.cwd().resolve()
BASELINE_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = BASELINE_DIR.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(BASELINE_DIR))

# Project modules
from config import get_dataset_paths, get_split_path_incremental
from src.dataset import DataModuleParasite
from src import utils
from src import models
from src import trainer

## Egg Dataset

In [ ]:
# Eggs dataset configuration
CONFIG_EGG = {
    'dataset_name': 'eggs',
    'split': [1, 2, 3],
    'percentage': [1, 5, 50, 100], #[1, 5, 25, 50, 75, 100]
    'num_classes': 9,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Setup for training 
config = CONFIG_EGG
dataset_name = config['dataset_name']
type_model = 'squeezenet'
image_size = config['image_size']

transforms_egg = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_egg)


### Training from Scratch

This section trains SqueezeNet on the eggs dataset using random initialization.

In [ ]:
# Training mode specific settings
model_name = 'squeezenet_scratch'
pre_trained = False
path = f"{type_model}/{model_name}/{dataset_name}"

# Create the model
model_scratch, _, _, _ = models.create_model(
    type_model=type_model,
    description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt",
    config=config,
    pre_trained=pre_trained
)

# Model complexity summary
input_size = (1, 3, config['image_size'], config['image_size'])
n_flops, n_params = utils.count_flops(model_scratch, input_size=input_size)
print(f"Model: {model_name} | Dataset: {dataset_name}")
print(f"FLOPs: {n_flops}")
print(f"Parameters: {n_params}")

In [ ]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

In [ ]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

### Fine-Tuning from ImageNet

This section fine-tunes SqueezeNet on the eggs dataset using ImageNet-pretrained weights.

In [ ]:
# Fine-tune model
model_name = 'squeezenet_pretrained'
pre_trained = True
path = f"{type_model}/{model_name}/{dataset_name}"

In [ ]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

In [ ]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

## Cyst Dataset

In [ ]:
# Cyst dataset configuration
CONFIG_CYST = {
    'dataset_name': 'cysts',
    'split': [1, 2, 3],
    'percentage': [1, 5, 50, 100], #[1, 5, 25, 50, 75, 100],
    'num_classes': 7,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Setup for training 
config = CONFIG_CYST
dataset_name = config['dataset_name']
type_model = 'squeezenet'
image_size = config['image_size']

transforms_cyst = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_cyst)


### Training from Scratch

This section trains SqueezeNet on the cyts dataset using random initialization.

In [ ]:
# Training mode specific settings
model_name = 'squeezenet_scratch'
pre_trained = False
path = f"{type_model}/{model_name}/{dataset_name}"

# Create the model
model_scratch, _, _, _ = models.create_model(
    type_model=type_model,
    description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt",
    config=config,
    pre_trained=pre_trained
)

# Model complexity summary
input_size = (1, 3, config['image_size'], config['image_size'])
n_flops, n_params = utils.count_flops(model_scratch, input_size=input_size)
print(f"Model: {model_name} | Dataset: {dataset_name}")
print(f"FLOPs: {n_flops}")
print(f"Parameters: {n_params}")

In [ ]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

In [ ]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

### Fine-Tuning from ImageNet

This section fine-tunes SqueezeNet on the cysts dataset using ImageNet-pretrained weights.

In [ ]:
# Fine-tune model
model_name = 'squeezenet_pretrained'
pre_trained = True
path = f"{type_model}/{model_name}/{dataset_name}"

In [ ]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

In [ ]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

## Larvae Dataset

In [ ]:
# Larvae dataset configuration
CONFIG_LARVAE = {
    'dataset_name': 'larvae',
    'split': [1,2,3],
    'percentage': [1, 5, 50, 100], #[1, 5, 25, 50, 75, 100],
    'num_classes': 2,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Setup for training 
config = CONFIG_LARVAE
dataset_name = config['dataset_name']
type_model = 'squeezenet'
image_size = config['image_size']

transforms_larvae = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_larvae)


### Training from Scratch

This section trains SqueezeNet on the larvae dataset using random initialization.

In [ ]:
# Training mode specific settings
model_name = 'squeezenet_scratch'
pre_trained = False
path = f"{type_model}/{model_name}/{dataset_name}"

# Create the model
model_scratch, _, _, _ = models.create_model(
    type_model=type_model,
    description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt",
    config=config,
    pre_trained=pre_trained
)

# Model complexity summary
input_size = (1, 3, config['image_size'], config['image_size'])
n_flops, n_params = utils.count_flops(model_scratch, input_size=input_size)
print(f"Model: {model_name} | Dataset: {dataset_name}")
print(f"FLOPs: {n_flops}")
print(f"Parameters: {n_params}")

In [ ]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)

In [ ]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

### Fine-Tuning from ImageNet

This section fine-tunes SqueezeNet on the larvae dataset using ImageNet-pretrained weights.

In [ ]:
# Fine-tune model
model_name = 'squeezenet_pretrained'
pre_trained = True
path = f"{type_model}/{model_name}/{dataset_name}"

In [ ]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

In [ ]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)